# Mixed Signals V2X - Training Pipeline

Cooperative 3D object detection (car, bike, pedestrian) on the Mixed Signals V2X
dataset. This notebook covers the pipeline used to produce the submitted model:
environment setup, dataset preparation, training, and evaluation.

The starter kit (`Mixed-Signals-Dataset/`, an OpenCOOD fork) expects the older
V2X_dataset-v0.4 label format. The current HuggingFace release ships flat
per-frame `.txt` labels instead, so `msig_compat/` bridges the two - see
`msig_compat/dataset_patch.py` for details.


In [1]:
%load_ext autoreload
%autoreload 2

## Setup

In [2]:
import os
import sys
import subprocess
import yaml

REPO_ROOT = os.getcwd() if os.path.isdir("Mixed-Signals-Dataset") else os.path.dirname(os.getcwd())  # notebook lives at the repo root
DS_ROOT = os.path.join(REPO_ROOT, "Mixed-Signals-Dataset")

if os.path.isdir("Mixed-Signals-Dataset"):
    os.chdir("Mixed-Signals-Dataset")

sys.path.insert(0, REPO_ROOT)
sys.path.insert(0, DS_ROOT)

from msig_compat.dataset_patch import apply_all_patches
apply_all_patches()

print("cwd:", os.getcwd())


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


cwd: /home/nitin/Nitin/LaneDetection/v2x-cooperative-detection/Mixed-Signals-Dataset


## Model

Intermediate (attention) fusion PointPillars, three classes. Config:
`opencood/hypes_yaml/msig_pillar_attnfuse_local.yaml`.

- Voxel size 0.2m, pillar encoder + BEV backbone, multi-scale attention fusion
  across nodes, shared detection head for all three classes.
- Held-out evaluation split: sequences 31, 32, 33, 34, 35, 37 (6 of 29
  downloaded sequences), the rest used for training.
- Adam, lr 5e-4, multistep decay at epochs 50/80, gamma 0.3.


In [3]:
HYPES_YAML = "opencood/hypes_yaml/msig_pillar_attnfuse_local.yaml"
FINAL_MODEL_DIR = "opencood/logs/final_model"

with open(os.path.join(FINAL_MODEL_DIR, "config.yaml")) as f:
    cfg = yaml.load(f, Loader=yaml.Loader)

print("lr:            ", cfg["optimizer"]["lr"])
print("lr_scheduler:  ", cfg["lr_scheduler"])
print("epoches:       ", cfg["train_params"]["epoches"])


lr:             0.0005
lr_scheduler:   {'core_method': 'multistep', 'gamma': 0.3, 'step_size': [50, 80]}
epoches:        100


## Training

```bash
cd Mixed-Signals-Dataset
PYTHONPATH=<repo_root>:. <venv_python> ../msig_compat/run_preprocessing.py   # build gt_infos, once
PYTHONPATH=<repo_root>:. <venv_python> ../msig_compat/run_train.py \
    --hypes_yaml opencood/hypes_yaml/msig_pillar_attnfuse_local.yaml
```

To resume from a checkpoint, add `--model_dir opencood/logs/<run_name>`.

The submitted checkpoint (`opencood/logs/final_model/net_epoch50.pth`) was trained
for 50 epochs. Training for longer was still improving accuracy at the point it
was stopped, so further epochs would likely help.


## Evaluation

In [4]:
RUN_EVAL = False  # flip to True to actually run

nvidia_smi = subprocess.run(
    ["nvidia-smi", "--query-gpu=memory.used,memory.total,utilization.gpu", "--format=csv,noheader"],
    capture_output=True, text=True
).stdout.strip()
print("nvidia-smi:", nvidia_smi)

if RUN_EVAL:
    sys.argv = [
        "inference.py",
        "--model_dir", FINAL_MODEL_DIR,
        "--fusion_method", "intermediate",
    ]
    from opencood.tools import inference
    inference.main()

    with open(os.path.join(FINAL_MODEL_DIR, "eval.yaml")) as f:
        eval_results = yaml.safe_load(f)
    print(f"\n{'class':<12}{'AP@0.3':>8}{'AP@0.5':>8}{'AP@0.7':>8}")
    for cls in ["car", "bike", "pedestrian"]:
        print(f"{cls:<12}{eval_results[f'{cls}_ap30']:>8.2f}"
              f"{eval_results[f'{cls}_ap_50']:>8.2f}{eval_results[f'{cls}_ap_70']:>8.2f}")
else:
    print("\nRUN_EVAL is False - skipping.")


nvidia-smi: 3669 MiB, 16376 MiB, 0 %

RUN_EVAL is False - skipping.


## Results

Held-out set: 6 sequences, 1791 frames.

| Class      | AP@0.3 | AP@0.5 | AP@0.7 |
|------------|--------|--------|--------|
| Car        | 0.85   | 0.85   | 0.85   |
| Bike       | 0.86   | 0.86   | 0.86   |
| Pedestrian | 0.63   | 0.50   | 0.17   |

Mean AP@0.5 across the three classes: 73.6%.

Pedestrian accuracy lags the other two classes, particularly at the stricter
IoU@0.7 threshold - car and bike stay flat across all three thresholds, which
points to a localization precision limit for small objects at the current
voxel/detection-grid resolution rather than a training issue. More training
epochs and a finer detection grid are both worth trying to close this gap
further.
